# SEG301 – Lab Text Preprocessing & Search Engine (bản nhạc Việt)
**File:** `Thanh_step1-2.ipynb` · **Phạm vi:** Step 1 (đọc & khám phá dữ liệu), Step 2 (làm sạch `clean_text` cho tiếng Việt), bộ truy vấn chấm điểm + thảo luận.

Dữ liệu: bảng `songs` trong `music_seg_merged_preprocesing.db` (SQLite) – lời bài hát crawl từ 4 website: Spotify, NhacCuaTui, Hợp Âm Chuẩn (hopamchuan.com), nhac.vn.

> Chạy notebook từ trên xuống dưới. Cell cuối ghi file `queries_step1-2.json` (bộ truy vấn + nhãn) để dùng cho Step 6-7.

In [1]:
import os, re, html, json, sqlite3, unicodedata
from collections import Counter
from urllib.parse import urlparse
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_columns", 30)

DB_NAME = "music_seg_merged_preprocesing.db"
DB_CANDIDATES = [DB_NAME, os.path.join("data", DB_NAME), os.path.join("/mnt/user-data/uploads", DB_NAME), os.path.join("/input", DB_NAME)]
DB_PATH = next(p for p in DB_CANDIDATES if os.path.exists(p))
print("DB:", DB_PATH)

DB: music_seg_merged_preprocesing.db


## Step 1 – Đọc bảng `songs` và khám phá dữ liệu

In [2]:
con = sqlite3.connect(DB_PATH)
print("Các bảng trong DB:", [r[0] for r in con.execute("select name from sqlite_master where type='table'")])

df_all = pd.read_sql("select * from songs", con)
con.close()

# 'website' lấy từ domain của url
df_all["site"] = df_all["url"].map(lambda u: urlparse(u).netloc.replace("www.", ""))

print("Số dòng, số cột:", df_all.shape)          # (rows, cols) – chưa tính cột site tự thêm
print("Các cột:", df_all.columns.tolist())
print()
print(df_all.dtypes)
df_all.head(3)

Các bảng trong DB: ['pages', 'sqlite_sequence', 'links', 'songs']
Số dòng, số cột: (2765, 9)
Các cột: ['id', 'url', 'title', 'artist', 'album', 'genre', 'lyrics', 'crawled_at', 'site']

id             int64
url           object
title         object
artist        object
album         object
genre         object
lyrics        object
crawled_at    object
site          object
dtype: object


,id,url,title,artist,album,genre,lyrics,crawled_at,site
0,1,https://www.nhaccuatui.com/song/ZGqkPWpPCsBH,SỚM MUỘN THÌ,"ANH TRAI ""SAY HI"", Hustlang Robber, Nhâm Phương Nam, Mason Nguyen, Jaysonlei, Khoi Vu,...",None,None,Baby à anh biết là\nSớm muộn gì thì\nMình cũng sẽ quay trở về\nBên nhau mà phải không\...,2026-10-12 08:24:39,nhaccuatui.com
1,2,https://www.nhaccuatui.com/song/k2TvHW1YQGPI,Pin Dự Phòng,"Dương Domic, Lou Hoàng",None,None,Có những lúc giống như tình yêu\nCó những lúc thấy tim mình phiêu\nBên nhau từ lâu mà ...,2026-10-12 08:25:15,nhaccuatui.com
2,3,https://www.nhaccuatui.com/song/r27tf4MELOuA,E Là Không Thể,Anh Quân,None,None,Ai cũng từng tin có phép màu.\nNên cứ để mặc tình yêu hóa thành đậm sâu\nXuân thì ngườ...,2026-10-12 08:25:27,nhaccuatui.com


In [3]:
# --- Giá trị thiếu: NULL và chuỗi rỗng/chỉ có khoảng trắng ---
missing = pd.DataFrame({
    "NULL": df_all.isnull().sum(),
    "chuỗi rỗng": df_all.apply(lambda c: c.astype(str).str.strip().eq("").sum() if c.dtype == object else 0),
})
missing["% NULL"] = (missing["NULL"] / len(df_all) * 100).round(1)
print(missing)

df_all["has_lyrics"] = df_all["lyrics"].fillna("").str.strip().ne("")
print("\nSố bài có lời:", int(df_all.has_lyrics.sum()), "| không có lời:", int((~df_all.has_lyrics).sum()))

            NULL  chuỗi rỗng  % NULL
id             0           0     0.0
url            0           0     0.0
title          0           0     0.0
artist        13           0     0.5
album        567           0    20.5
genre       2585           0    93.5
lyrics       629           0    22.7
crawled_at     0           0     0.0
site           0           0     0.0

Số bài có lời: 2136 | không có lời: 629


In [4]:
# --- Số bài theo website + tỉ lệ có lời ---
by_site = df_all.groupby("site").agg(so_bai=("id", "size"), co_loi=("has_lyrics", "sum"))
by_site["thieu_loi"] = by_site.so_bai - by_site.co_loi
by_site["% co_loi"] = (by_site.co_loi / by_site.so_bai * 100).round(1)
by_site = by_site.sort_values("so_bai", ascending=False)
print(by_site)
print("\nGhi chú: các cột album/genre thiếu rất nhiều (genre chỉ hopamchuan/nhac.vn có) -> không dùng làm nhãn.")

                  so_bai  co_loi  thieu_loi  % co_loi
site                                                 
open.spotify.com    2194    1566        628      71.4
nhaccuatui.com       322     321          1      99.7
hopamchuan.com       227     227          0     100.0
nhac.vn               22      22          0     100.0

Ghi chú: các cột album/genre thiếu rất nhiều (genre chỉ hopamchuan/nhac.vn có) -> không dùng làm nhãn.


In [5]:
# --- Kiểm tra trùng ---
songs = df_all[df_all.has_lyrics].copy()          # chỉ xét bài có lời
print("url trùng:", df_all.url.duplicated().sum(), "(cột url là UNIQUE nên luôn = 0)")

# 1) Trùng NGUYÊN VĂN lời (raw)
dup_mask = songs.duplicated("lyrics", keep=False)
grp = (songs[dup_mask].groupby("lyrics")
       .agg(n=("id", "size"), ids=("id", list), n_title=("title", "nunique"),
            n_site=("site", "nunique"), title=("title", "first")))
print(f"\nLời trùng nguyên văn: {dup_mask.sum()} dòng thuộc {len(grp)} nhóm "
      f"(nếu xoá trùng sẽ mất {int(grp.n.sub(1).sum())} dòng)")
print("Kích thước nhóm:", grp.n.value_counts().sort_index().to_dict())
print("Nhóm có >1 tiêu đề khác nhau (cùng lời, khác tên bài):", int((grp.n_title > 1).sum()))
print("Nhóm nằm trên >1 website:", int((grp.n_site > 1).sum()))

# 2) Trùng (title, artist) sau khi chuẩn hoá nhẹ
def norm_key(s):
    s = html.unescape(unicodedata.normalize("NFC", str(s))).lower()
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", s)).strip()
songs["ta_key"] = songs.title.map(norm_key) + " | " + songs.artist.fillna("").map(norm_key)
print("\nDòng có cùng (title, artist):", int(songs.duplicated("ta_key", keep=False).sum()))
print("…trong đó cùng (title, artist) nhưng lời KHÁC nhau:",
      int(songs.groupby("ta_key").lyrics.nunique().gt(1).sum()), "nhóm")

grp.sort_values("n", ascending=False).head(8)[["n", "n_site", "n_title", "title", "ids"]]

url trùng: 0 (cột url là UNIQUE nên luôn = 0)

Lời trùng nguyên văn: 234 dòng thuộc 105 nhóm (nếu xoá trùng sẽ mất 129 dòng)
Kích thước nhóm: {2: 84, 3: 19, 4: 1, 5: 1}
Nhóm có >1 tiêu đề khác nhau (cùng lời, khác tên bài): 30
Nhóm nằm trên >1 website: 0

Dòng có cùng (title, artist): 347
…trong đó cùng (title, artist) nhưng lời KHÁC nhau: 89 nhóm


,n,n_site,n_title,title,ids
lyrics,,,,,
Nghệ sĩ\n\nTrapsoul2021 người theo dõi\n\nTheo dõi,5,1,5,On Empty,"[2713, 2714, 2715, 2716, 2717]"
"Ah-ah-ah-a-a-á-ah-ạ\nA-a-à-ah-à-a-a-á-ah-ạ\nNgười theo hương hoa mây mù giăng lối (ah-ah)\nLàn sương khói phôi phai đưa bước ai xa rồi (xa rồi)\nĐơn côi mình ta vấn vương\nHồi ức trong men say chiều mưa buồn\nNgăn giọt lệ ngừng khiến khóe mi sầu bi í-i-ĩ-i\nĐường xưa nơi cố nhân từ giã biệt ly (cánh hoa rụng rời, rụng rời)\nPhận duyên mong manh rẽ lối trong mơ ngày tương phùng (oh-oh)\nTiếng khóc cuốn theo làn gió bay (ah-ah)\nThuyền ai qua sông lỡ quên vớt ánh trăng tàn nơi này (ah-ah-á)\nTrống vắng bóng ai dần hao gầy (á-ah-ah-ua-ua-ah)\nLòng ta xin nguyện khắc ghi trong tim tình nồng mê say\nMặc cho tóc mây vương lên đôi môi cay\nBâng khuâng mình ta lạc trôi giữa đời (a-ạ-á)\nTa lạc trôi giữa trời ah-à-ạ\nĐôi chân lang thang về nơi đâu (đâu)\nBao yêu thương giờ nơi đâu (đâu)\nCâu thơ tình xưa vội phai mờ\nTheo làn sương tan biến trong cõi mơ (cõi mơ)\nMưa bụi vương trên làn mi mắt (mắt)\nNgày chia lìa hoa rơi buồn hiu hắt (hắt)\nTiếng đàn ai thêm sầu tương tư lặng\nMình trong chiều hoàng hôn tan vào lời ca (ooh)\nLối mòn đường vắng một mình ta (oá)\nNắng chiều vàng úa nhuộm ngày qua (oá)\nXin đừng quay lưng xoá (eh)\nĐừng mang câu hẹn ước kia rời xa (oá)\nYên bình nơi nào đây-eh\nChôn vùi theo làn mây (yeah yeah yeah là-la-la-la-la-la-lá-la la-lạ)\nNgười theo hương hoa mây mù giăng lối (ah-ah)\nLàn sương khói phôi phai đưa bước ai xa rồi (áh-áh)\nĐơn côi mình ta vấn vương\nHồi ức trong men say chiều mưa buồn\nNgăn giọt lệ ngừng khiến khóe mi sầu bi í-i-ĩ-i\nTừ xưa nơi cố nhân từ giã biệt ly (cánh hoa rụng rời, rụng rời)\nPhận duyên mong manh rẽ lối trong mưa ngày tương phùng (oh-oh)\nTiếng khóc cuốn theo làn gió bay (ah-ah)\nThuyền ai qua sông lỡ quên vớt ánh trăng tàn nơi này (ah-ah-á)\nTrống vắng bóng ai dần hao gầy (á-ah-ah-ua-ua-ah)\nLòng ta xin nguyện khắc ghi trong tim tình nồng mê say\nMặc cho tóc mây vương lên đôi môi cay\nBâng khuâng mình ta lạc trôi giữa đời (a-á)\nTa lạc trôi giữa trời ah-à-ah-ạ\nTa lạc trôi lạc trôi (lạc trôi)\nTa lạc trôi giữa đời (lạc trôi giữa đời, lạc trôi giữa đời)\nLạc trôi giữa trời (lạc trôi giữa trời, lạc trôi giữa trời)\nYeah ah-ah-á-á-a-a-ạ-á-a-ạ\nAh-á-ah-áh-á-a-à-a-à\nTa đang lạc nơi nào (nơi nào, nơi nào, oá oá oá)\nTa đang lạc nơi nào (oá oá oá-eh)\nLối mòn đường vắng một mình ta\n(Ta đang lạc nơi nào la-la-la-la-la-la-la-la oá oá oá)\nNắng chiều vàng úa nhuộm ngày qua\n(Ta đang lạc nơi nào oh-oh-ôi-ôi-ôi-ôi)",4,1,1,Lạc Trôi,"[536, 1128, 1145, 1278]"
"Bờ vai ai mang vác tấm hàng?\nMồ hôi tuôn rơi những tháng năm hao gầy\nVì tình yêu thương con, cha đánh đổi\nChạy buôn thâu đêm đi tuyến Sơn La, Thái Bình\n\nCó những ngày mùa đông lạnh thấu nhưng cha vẫn chưa về\nMẹ cười nhưng con biết mẹ lo lắm, cha ơi\nXuân chỉ về khi con nghe thấy tiếng xe quen thuộc\nTuổi thơ ơi, nhớ lắm bánh cáy theo tôi lớn khôn (theo tôi lớn khôn, theo tôi lớn khôn)\n\nThời gian trôi, con lớn dần\nCon biết ước mơ của con là gì\nChạy theo đam mê, con sợ con quên đi quê hương\nQuên mất một điều tuyệt vời\nMãi là người con Thái Bình, là con của mẹ\n\nNhớ đường xưa, dòng sông uốn quanh mộng mơ\nĐùa vui dưới mưa tình yêu, nơi thắm lên con tim này, oh-ooh-ooh-ooh-ooh\nNhớ lời ru nuôi con lớn lên, bài ca gửi vào trong gió êm đềm\nXin mây đưa đi về phương xa\n\nOh-oh-oh-ah-whoo\nOh-oh-oh-ah-whoo\nOh-oh-oh-ah-whoo\nOh-ah-ah-ah-ah-ah, oh-oh-oh\n\nAh-ah-ah\nAh-ah-ah\n\nCó những ngày mùa đông lạnh thấu nhưng cha vẫn chưa về\nMẹ cười nhưng con biết mẹ lo lắm, cha ơi\nXuân chỉ về khi con nghe thấy tiếng xe quen thuộc\nTuổi thơ ơi, nhớ lắm bánh cáy theo tôi lớn khôn, hmm-mm\n\nAh-ah-ah-ah-huh\nThời gian trôi (ah-ah-ah-ah-huh), con lớn dần (ah-ah-ah-ah-huh)\nCon biết ước mơ của con là gì\nChạy theo đam mê (ah-ah-ah-ah-huh), con sợ con quên đi quê hương (ah-ah-ah-ah-huh)\nQuên mất một điều tuyệt vời (ah-ah-ah-ah-huh)\nCon mãi là người con Thái Bình, là con của mẹ\n\nNhớ đường xưa, dòng sôn

In [6]:
# Ví dụ: cùng lời nhưng KHÁC tên bài / nghệ sĩ  (bản remix, lo-fi, cover, đăng lại...)
ex = grp[grp.n_title > 1].head(4)
for lyr, r in ex.iterrows():
    print("-" * 80)
    print(songs[songs.lyrics == lyr][["id", "site", "title", "artist"]].to_string(index=False))

# Ví dụ: cùng tên bài nhưng KHÁC bài (lời khác hẳn) -> không được gộp theo tên
print("\n" + "=" * 80)
print(songs[songs.title.str.lower().str.strip().isin(["tìm em", "xin lỗi", "cô ta"])]
      [["id", "site", "title", "artist"]].sort_values(["title", "id"]).to_string(index=False))

--------------------------------------------------------------------------------
  id             site                          title                                                   artist
1740 open.spotify.com                         SO ĐẬM EM XINH "SAY HI", Phương Ly, Vũ Thảo My, Châu Bùi, Muộii
1758 open.spotify.com SO ĐẬM - HaNoi Concert Version EM XINH "SAY HI", Phương Ly, Vũ Thảo My, Châu Bùi, Muộii
--------------------------------------------------------------------------------
 id             site              title        artist
529 open.spotify.com CÓ CHẮC YÊU LÀ ĐÂY Sơn Tùng M-TP
686 open.spotify.com Có Chắc Yêu Là Đây Sơn Tùng M-TP
--------------------------------------------------------------------------------
  id             site                            title                  artist
2573 open.spotify.com Nếu Như Anh Đến - Remix DJ BNUTS Văn Mai Hương, DJ BNUTS
2583 open.spotify.com Nếu Như Anh Đến - Dj Bnuts Remix           Văn Mai Hương
----------------------------

### Có nên xoá bài trùng lời không?

**Kết luận: KHÔNG xoá trong corpus (không dùng `drop_duplicates` như lab BBC).** Chỉ *đánh dấu* nhóm trùng (`dup_group`) để dùng khi cần.

1. **Mỗi dòng là một tài liệu hợp lệ có `id` riêng.** Bộ nhãn của lab là *"mọi phiên bản của đúng bài đó"* (Spotify đăng 3–4 bản của cùng một bài, NhacCuaTui có hàng chục remix/lo-fi/cover của "Thiệp Hồng Sai Tên", "Nắng Dưới Chân Mây"…). Nếu xoá trùng, ta xoá luôn các tài liệu *liên quan* → Recall@K bị tính sai (mẫu số `|R_q|` thay đổi theo cách xoá).
2. **"Trùng" ở đây không đồng nghĩa "thừa".** Cùng lời nhưng khác tên/nghệ sĩ (remix, lo-fi, cover) là các đối tượng khác nhau; còn *cùng tên bài nhưng lời khác nhau* (ví dụ "Tìm Em", "Xin Lỗi", "Cô Ta") thì tuyệt đối không được gộp theo tên. Ngoài ra, trùng **nguyên văn** chỉ xảy ra *trong cùng một website* (số nhóm trải qua >1 website = 0): các bản của cùng một bài ở web khác nhau là *trùng gần đúng* (chép lời khác nhau, lệch rác/nhãn đoạn), nên `drop_duplicates` cũng không bắt được chúng – cần so khớp mờ (shingle/Jaccard), xem phần nhãn bên dưới.
3. **Khác với BBC:** ở BBC trùng bài là lỗi crawl và nhãn `category` cần nhất quán; ở đây trùng là đặc tính của dữ liệu nhạc, và `url` đã UNIQUE nên không có dòng trùng thật sự.
4. **Rủi ro nếu giữ nguyên:** top-K có thể bị chiếm bởi nhiều bản copy của *một* bài → Precision@K "đẹp giả", IDF/BM25 bị lệch (tần suất tài liệu bị phồng). Cách xử lý thay vì xoá: (a) báo cáo thêm metric ở mức "bài hát" (gộp kết quả theo `dup_group`/work id), (b) hoặc khử trùng *ở bước hiển thị kết quả*, không ở corpus.
5. Nhóm trùng lớn nhất ở bảng trên (5 dòng "Nghệ sĩ … người theo dõi") thực chất là **trang không có lời** (chỉ còn khối nghệ sĩ) – loại bỏ chúng là hợp lý, nhưng vì *không có lời*, không phải vì trùng.
6. Chỉ **loại các dòng không có lời** (≈ 629 dòng NULL, chủ yếu Spotify) khỏi corpus tìm kiếm vì không có gì để index – nhưng giữ lại trong `df_all` để biết tỉ lệ thiếu.

## Step 2 – `clean_text` cho tiếng Việt

Quan sát từng website (xem bảng): lời bài hát bị lẫn nhiều rác crawl, mỗi nguồn một kiểu.

| Nguồn | Rác cần bỏ |
|---|---|
| **NhacCuaTui** | đầu: `Đăng bởi<tên>`; đuôi: `Xem thêm / Nghệ sĩ / <tên> / 12345 người theo dõi / Theo dõi` (bản tiếng Anh: `Artist / followers / Follow / Show more`) |
| **Hợp Âm Chuẩn** | dòng đầu về hợp âm: `Tone gốc D: Capo 2`, `Nhịp: 4/4 …`, `Intro: x2`, `Vòng hợp âm: …`, `Có thể đánh …` |
| **nhac.vn** | `|`, `Hợp Âm Guitar`, `Lời bài hát` ở đầu; `Xem hết` ở cuối |
| **Spotify** | dòng tiêu đề `TÊN BÀI - NGHỆ SĨ` + `Sáng tác: …`; dòng credit cuối (`NetEase - nct.`); ký tự `♪`; chữ lạ kiểu Cyrillic/Hy Lạp giả dạng Latin (anti-scrape: `Ϲó`, `naу`) |
| **Chung** | HTML entity/tag, URL (vd `Lyric: … (https://thienkim.top)`), header `Bài hát:/Ca sĩ:/Lyric:`, nhãn đoạn `[Verse 1]`, `[Chorus]:`, `ĐK:`, `Hook:`, `Rap:`, nhãn người hát `[Thắng:]`, đánh số đoạn `1.`, ký hiệu `* ^ ~` |

In [7]:
# ===================== clean_text (tiếng Việt) =====================
# Ký tự "trắng" lạ -> space ; \r -> newline
SPACE_CHARS = dict.fromkeys(map(ord, "\u00a0\u2005\u2009\u200a\u202f\u205f\u3000\u200b\u200c\u200d\ufeff"), " ")
SPACE_CHARS[ord("\r")] = "\n"
# Chữ Cyrillic/Hy Lạp trông giống Latin (Spotify chèn để chống scrape) -> Latin
HOMOGLYPHS = str.maketrans({"у": "y", "е": "e", "о": "o", "а": "a", "с": "c", "р": "p", "х": "x", "і": "i", "ѕ": "s", "Ϲ": "C"})
RE_CJK_KO = re.compile(r"[\uac00-\ud7af\u3040-\u30ff\u4e00-\u9fff]")   # có tiếng Hàn/Nhật/Trung -> không đụng Cyrillic

SECTION = (r"verse|vers|ver|chorus|pre[- ]?chorus|post[- ]?chorus|bridge|intro|outro|hook|rap|coda|interlude|"
           r"instrumental|refrain|solo|break|drop|đk|dk|điệp khúc|phiên khúc|đoạn")

RE_TAG        = re.compile(r"<[^>]+>")
RE_URL        = re.compile(r"https?://\S+|www\.\S+", re.I)
RE_BRACKET    = re.compile(r"\[[^\]\n]{0,120}\]:?")                       # [Verse 1] [Thắng:] [Chorus: A, B & C]
RE_LABEL_LINE = re.compile(rf"^[\[\(]?\s*(?:{SECTION})(?:\s*\d+)?\s*[\]\)]?\s*:?\s*[\]\)]?$", re.I)   # cả dòng chỉ là nhãn
RE_LABEL_PREF = re.compile(rf"^[\[\(]?\s*(?:{SECTION})(?:\s*\d+)?\s*[\]\)]?\s*:\s*[\]\)]?\s*", re.I)  # "ĐK: lời..." -> "lời..."
RE_META_HDR   = re.compile(r"^(?:bài hát|ca sĩ|trình bày|thể hiện|sáng tác|nhạc sĩ|lyric|lời|lời bài hát|nhạc)\s*:", re.I)
RE_NCT_MARK   = re.compile(r"^(?:xem thêm|show more|nghệ sĩ|artist)$", re.I)
RE_NCT_FOLLOW = re.compile(r"(?:người theo dõi|followers?)\s*$|^(?:theo dõi|follow)$", re.I)
RE_HAC_HDR    = re.compile(r"\b(?:tone|capo)\b|^(?:nhịp|intro|outro|vòng hợp âm|vòng hòa thanh|cách đánh)\s*[:\-]|^\(?có thể đánh", re.I)
RE_SPOTIFY_CREDIT = re.compile(r"^(?:netease\s*-\s*)?nct\.?$", re.I)


def site_of(url):
    return urlparse(url).netloc.replace("www.", "")


def _first_nonempty(lines, k):
    return [i for i, l in enumerate(lines) if l.strip()][:k]


def strip_site_boilerplate(text, site):
    # Xoá rác theo từng website (làm theo DÒNG, trước khi dồn về 1 dòng).
    lines = text.split("\n")
    if site == "nhaccuatui.com":
        lines = [l for l in lines if not re.match(r"^\s*đăng bởi", l, re.I)]
        n = len(lines)
        cut = next((i for i in range(max(0, n - 30), n) if RE_NCT_MARK.match(lines[i].strip())), None)
        if cut is not None:
            lines = lines[:cut]                      # bỏ toàn bộ đuôi "Xem thêm / Nghệ sĩ / …"
        lines = [l for l in lines if not RE_NCT_FOLLOW.search(l.strip())]
    elif site == "nhac.vn":
        lines = [l for l in lines if l.strip() not in {"|", "Hợp Âm Guitar", "Lời bài hát"}]
        while lines and not lines[-1].strip():
            lines.pop()
        if lines:
            lines[-1] = re.sub(r"\s*xem hết\s*$", "", lines[-1], flags=re.I)
    elif site == "hopamchuan.com":
        drop = {i for i in _first_nonempty(lines, 8) if RE_HAC_HDR.search(lines[i].strip())}   # chỉ xét vùng header
        lines = [l for i, l in enumerate(lines) if i not in drop]
    elif site == "open.spotify.com":
        idx = _first_nonempty(lines, 3)
        drop = set()
        for k, i in enumerate(idx):
            if RE_META_HDR.match(lines[i].strip()):
                drop.add(i)
                if k == 1:
                    drop.add(idx[0])                 # dòng "TÊN BÀI - NGHỆ SĨ" ngay trên "Sáng tác:"
        lines = [l for i, l in enumerate(lines) if i not in drop]
        while lines and (not lines[-1].strip() or RE_SPOTIFY_CREDIT.match(lines[-1].strip())):
            lines.pop()                              # "NetEase - nct." / "nct."
    # Header chung "Bài hát: … / Ca sĩ: … / Lyric: …" (chỉ trong 10 dòng đầu)
    drop = {i for i in _first_nonempty(lines, 10) if RE_META_HDR.match(lines[i].strip())}
    return "\n".join(l for i, l in enumerate(lines) if i not in drop)


def strip_labels_and_markers(text):
    out = []
    for l in text.split("\n"):
        s = l.strip()
        if not s:
            continue
        if RE_LABEL_LINE.match(s):
            continue
        if s.endswith(":") and len(s) <= 25 and len(s.split()) <= 3:   # "Nam:", "Rap (Richie):" – nhãn người hát
            continue
        if re.match(r"^\[[^\]]{0,10}$", s):                            # nhãn bị cắt dở: "[Đ"
            continue
        s = RE_BRACKET.sub(" ", s)                       # [Verse 1], [Thắng:], [ĐK]:
        s = RE_LABEL_PREF.sub("", s)                     # "ĐK: …", "Chorus: …"
        s = re.sub(r"^\s*\d{1,2}\s*[.)]\s*", "", s)      # đánh số đoạn "1. …"
        s = re.sub(r"^\s*[*^~♪]+\s*|\s*[*^~♪]+\s*$", " ", s).strip()   # ký hiệu lặp/ngắt
        if s:
            out.append(s)
    return "\n".join(out)


def clean_text(text, site=None):
    # Làm sạch lời bài hát tiếng Việt -> 1 dòng, chữ thường, GIỮ dấu tiếng Việt, số, gạch nối, dấu nháy.
    if text is None or isinstance(text, float):
        return ""
    text = unicodedata.normalize("NFC", str(text))       # 1) Unicode NFC (dấu dựng sẵn)
    text = html.unescape(text)                           # 2) &amp; &#039; -> & '
    text = text.translate(SPACE_CHARS)                   # 3) NBSP, zero-width... -> space
    text = RE_TAG.sub(" ", text)                         # 4) HTML tag
    text = RE_URL.sub(" ", text)                         # 5) URL
    if re.search(r"[\u0400-\u04ff\u03f9]", text) and not RE_CJK_KO.search(text):
        text = text.translate(HOMOGLYPHS)                # 6) chữ giả Latin
    if site:
        text = strip_site_boilerplate(text, site)        # 7) rác riêng từng web
    text = strip_labels_and_markers(text)                # 8) nhãn [Verse]/ĐK:, đánh số, ký hiệu
    text = re.sub(r"\s+", " ", text)                     # 9) chuẩn hoá khoảng trắng
    return unicodedata.normalize("NFC", text.strip().lower())   # 10) chữ thường (Đ -> đ ok)


# Biến thể thí nghiệm (KHÔNG ghi đè clean_text, giống lab BBC)
def remove_punctuation(text):
    # Lưu ý: KHÔNG dùng [^a-zA-Z0-9\s] như lab BBC – sẽ xoá sạch chữ có dấu tiếng Việt. \w trong Unicode giữ được chữ có dấu.
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]|_", " ", text)).strip()


def fold_accents(text):
    # Bỏ dấu tiếng Việt (đ -> d) để khớp truy vấn gõ không dấu như "lac troi".
    text = unicodedata.normalize("NFD", text.replace("đ", "d").replace("Đ", "D"))
    return unicodedata.normalize("NFC", "".join(c for c in text if unicodedata.category(c) != "Mn"))


# smoke test
assert clean_text("Verse 1:\nHà&nbsp;Nội &amp; Sài Gòn http://x.vn\n[Chorus]\nM-TP don&#039;t", "open.spotify.com") == "hà nội & sài gòn m-tp don't"
assert fold_accents("Lạc Trôi, Đừng đi") == "Lac Troi, Dung di"
print("clean_text OK")

clean_text OK


In [8]:
df_all["clean_text"] = [clean_text(t, s) for t, s in zip(df_all["lyrics"], df_all["site"])]
df_all["raw_len"] = df_all["lyrics"].fillna("").str.len()
df_all["clean_len"] = df_all["clean_text"].str.len()

empty_after = df_all[df_all.has_lyrics & (df_all.clean_len == 0)]
print("Bài có lời nhưng sau làm sạch rỗng (chỉ còn khối 'Nghệ sĩ…'):", len(empty_after))
print(empty_after[["id", "site", "title", "raw_len"]].to_string(index=False))

# corpus dùng cho Step 3+: có lời thật sự
df_all["in_corpus"] = df_all.clean_len > 0
corpus = df_all[df_all.in_corpus].reset_index(drop=True)
print("\nCorpus:", corpus.shape, "| theo web:", corpus.site.value_counts().to_dict())

stat = df_all[df_all.in_corpus].groupby("site").agg(raw_len=("raw_len", "mean"), clean_len=("clean_len", "mean")).round(0)
stat["giam_%"] = ((1 - stat.clean_len / stat.raw_len) * 100).round(1)
stat

Bài có lời nhưng sau làm sạch rỗng (chỉ còn khối 'Nghệ sĩ…'): 6
  id           site               title  raw_len
2710 nhaccuatui.com   Yerb Uno (118.00)       45
2713 nhaccuatui.com            On Empty       46
2714 nhaccuatui.com             Tonight       46
2715 nhaccuatui.com Eat Dat P Like This       46
2716 nhaccuatui.com                  48       46
2717 nhaccuatui.com      Ain’t Got Time       46

Corpus: (2130, 14) | theo web: {'open.spotify.com': 1566, 'nhaccuatui.com': 315, 'hopamchuan.com': 227, 'nhac.vn': 22}


,raw_len,clean_len,giam_%
site,,,
hopamchuan.com,1187.0,1180.0,0.6
nhac.vn,1205.0,1160.0,3.7
nhaccuatui.com,1885.0,1774.0,5.9
open.spotify.com,1717.0,1707.0,0.6


### So sánh raw vs clean (mỗi website 1 bài, tổng cộng 4 bài + 1 bài có nhãn đoạn)

In [9]:
def show_compare(song_id, head=260, tail=140):
    r = df_all[df_all.id == song_id].iloc[0]
    print("=" * 100)
    print(f"id={r.id} | {r.site} | {r.title} – {r.artist}")
    print("RAW  đầu :", repr(r.lyrics[:head]))
    print("RAW  cuối:", repr(r.lyrics[-tail:]))
    print("CLEAN đầu :", r.clean_text[:head])
    print("CLEAN cuối:", r.clean_text[-tail:])

for sid in [263, 2709, 501, 739, 2745]:     # NhacCuaTui(đuôi), HợpÂmChuẩn(Tone/Capo), nhac.vn(Lời bài hát), Spotify(Sáng tác), HợpÂmChuẩn(Tone+Vòng+Intro)
    show_compare(sid)

id=263 | nhaccuatui.com | An Thần – Low G, Thắng
RAW  đầu : 'Tại sao lúc vui em bao giờ cũng đang bận?\nLúc bên anh bao giờ cũng đang giận\nNói chung anh chỉ là thuốc an thần\nVới em anh chỉ là thuôc an thần\nNói em không nên dùng quá 2 lần\nNói em chỉ nên dùng lúc em cần\nVới em anh chỉ là thuốc an thần\nVới em anh chỉ là thằ'
RAW  cuối: 'hỉ là thuốc an thần\nVới em anh chỉ là thuốc an thần\nXem thêm\nNghệ sĩ\nLow G\n46615 người theo dõi\nTheo dõi\nThắng\n10767 người theo dõi\nTheo dõi'
CLEAN đầu : tại sao lúc vui em bao giờ cũng đang bận? lúc bên anh bao giờ cũng đang giận nói chung anh chỉ là thuốc an thần với em anh chỉ là thuôc an thần nói em không nên dùng quá 2 lần nói em chỉ nên dùng lúc em cần với em anh chỉ là thuốc an thần với em anh chỉ là thằ
CLEAN cuối: ùng quá 2 lần nói em chỉ nên dùng lúc em cần với em anh chỉ là thuốc an thần với em anh chỉ là thuốc an thần với em anh chỉ là thuốc an thần
id=2709 | hopamchuan.com | Chúng ta của hiện tại – Sơn Tùng M-TP
RAW  đầu : 'Tone

In [10]:
# Còn sót rác không? (đếm theo website, kỳ vọng = 0)
LEFTOVER = {
    "xem thêm/xem hết": r"xem thêm|xem hết|show more",
    "theo dõi/follow":  r"người theo dõi|\bfollowers?\b|\btheo dõi\b",
    "đăng bởi":         r"đăng bởi",
    "capo/tone":        r"\bcapo\b|tone gốc",
    "sáng tác:":        r"sáng tác\s*:",
    "[...] còn sót":    r"[\[\]]",
    "url/html":         r"https?://|www\.|</?\w+>|&\w+;|&#\d+;",
    "nhãn verse/chorus":r"\b(?:verse|chorus)\b\s*\d*\s*:",
}
left = pd.DataFrame({k: corpus.groupby("site").clean_text.apply(lambda s, p=p: int(s.str.contains(p, regex=True).sum()))
                     for k, p in LEFTOVER.items()})
left

,xem thêm/xem hết,theo dõi/follow,đăng bởi,capo/tone,sáng tác:,[...] còn sót,url/html,nhãn verse/chorus
site,,,,,,,,
hopamchuan.com,0,0,0,0,0,0,0,0
nhac.vn,0,0,0,0,0,0,0,0
nhaccuatui.com,0,1,0,0,0,0,0,0
open.spotify.com,0,3,0,0,0,0,0,0


In [11]:
# Cái gì còn sót thì xem để biết là lỗi hay là lời thật?
for name, pat in {"[...] còn sót": r"[\[\]]", "theo dõi/follow": r"người theo dõi|\bfollowers?\b|\btheo dõi\b", "capo/tone": r"\bcapo\b|tone gốc"}.items():
    m = corpus[corpus.clean_text.str.contains(pat, regex=True)]
    print(f"--- {name}: {len(m)} bài")
    for _, r in m.head(4).iterrows():
        mm = re.search(pat, r.clean_text)
        print(f"   id={r.id} {r.site}: …{r.clean_text[max(0, mm.start()-60): mm.end()+60]}…")

--- [...] còn sót: 0 bài
--- theo dõi/follow: 4 bài
   id=267 nhaccuatui.com: …ình yêu em chỉnh reel, dùng filter che điểm yếu dụ được anh theo dõi thì em nghĩ mình siêu? nah, chuyện đâu phải thế ok nhiệm vụ…
   id=566 open.spotify.com: …mơ của mình, từ ô cửa, sổ máy bay và anh biết, em vẫn luôn, theo dõi anh mỗi ngày để cho anh, có cảm giác, là em luôn ở đây ta đ…
   id=1155 open.spotify.com: …hú vui của một gã điên chỉ vì một câu nói, lòng đau nhói bỏ theo dõi, để lại trang cá nhân lẻ loi mang cả những chiếc ôm bỗng kh…
   id=1189 open.spotify.com: …ình yêu em chỉnh reel, dùng filter che điểm yếu dụ được anh theo dõi thì em nghĩ mình siêu? nah, chuyện đâu phải thế ok, nhiệm v…
--- capo/tone: 0 bài


### Bỏ số / gạch nối / dấu nháy / dấu câu thì mất gì?
Lab BBC cảnh báo: *"số và dấu câu có thể mang nghĩa"*. Ở đây đo trực tiếp trên corpus.

In [12]:
txt = corpus.clean_text

def count_tokens(pattern):
    c = Counter()
    for t in txt:
        c.update(re.findall(pattern, t))
    return c

num   = count_tokens(r"\b\d+[\w%]*\b")                  # 2019, 2k, 3, x2 ...
hyph  = count_tokens(r"\b\w+(?:-\w+)+\b")               # m-tp, oh-oh-oh, t-ĐK ...
apos  = count_tokens(r"\b\w+['’]\w+\b")                 # don't, i'm, lay's ...

for name, c in [("số / token chứa số", num), ("từ có gạch nối", hyph), ("từ có dấu nháy", apos)]:
    print(f"{name}: {sum(c.values())} lần xuất hiện, {len(c)} dạng khác nhau")
    print("   ví dụ phổ biến:", c.most_common(12))
print("\nSố bài có chứa 'm-tp':", int(txt.str.contains(r"\bm-tp\b").sum()), "| có 'mtp'/'m tp':", int(txt.str.contains(r"\bmtp\b|\bm tp\b").sum()))

số / token chứa số: 569 lần xuất hiện, 82 dạng khác nhau
   ví dụ phổ biến: [('1', 160), ('2', 59), ('3', 30), ('5', 29), ('10', 26), ('16', 25), ('3d', 17), ('15', 14), ('4', 12), ('20', 12), ('11', 10), ('7', 9)]
từ có gạch nối: 3136 lần xuất hiện, 846 dạng khác nhau
   ví dụ phổ biến: [('oh-oh', 112), ('m-tp', 90), ('ah-ah', 73), ('oh-oh-oh-oh', 56), ('oh-oh-oh', 56), ('uh-huh', 54), ('la-la-la-la', 52), ('ah-ah-ah-ah', 51), ('uh-uh-uh-uh-uh-uh-uh', 48), ('ooh-ooh', 42), ('ah-ah-ah', 36), ('uh-uh', 34)]
từ có dấu nháy: 3379 lần xuất hiện, 158 dạng khác nhau
   ví dụ phổ biến: [("don't", 572), ("i'm", 546), ("you're", 300), ("it's", 276), ("let's", 161), ("i'll", 159), ("can't", 92), ("ain't", 91), ("that's", 77), ('i’m', 76), ("we're", 68), ("won't", 68)]

Số bài có chứa 'm-tp': 39 | có 'mtp'/'m tp': 9


In [13]:
# Thử 3 kiểu bỏ dấu câu trên 1 câu thực tế
sample = "Sơn Tùng M-TP: don't say 2019, I'm lạc trôi! Tết x2 (oh-oh-oh)"
s = clean_text(sample)
print("clean_text            :", s)
print("regex của lab BBC     :", re.sub(r"\s+", " ", re.sub(r"[^a-zA-Z0-9\s]", " ", s)).strip(), "   <-- mất sạch chữ có dấu!")
print("remove_punctuation(\\w):", remove_punctuation(s))
print("fold_accents          :", fold_accents(s))

# Tác hại của regex ASCII của lab BBC trên cả corpus
ascii_only = txt.map(lambda t: re.sub(r"[^a-zA-Z0-9\s]", " ", t))
lost = 1 - ascii_only.str.replace(r"\s", "", regex=True).str.len().sum() / txt.str.replace(r"\s", "", regex=True).str.len().sum()
print(f"\nRegex [^a-zA-Z0-9\\s] làm mất {lost*100:.1f}% ký tự của corpus và băm vỡ từ tiếng Việt (vd 'lạc trôi' -> 'l c tr i').")

# Thêm 2 cột thí nghiệm, không ghi đè clean_text
corpus["clean_no_punct"] = corpus.clean_text.map(remove_punctuation)
corpus["clean_no_accent"] = corpus.clean_text.map(fold_accents)
print("\nvocab (tách theo khoảng trắng): clean_text =", len({w for t in corpus.clean_text for w in t.split()}),
      "| no_punct =", len({w for t in corpus.clean_no_punct for w in t.split()}),
      "| no_accent =", len({w for t in corpus.clean_no_accent for w in t.split()}))
print("(vocab tính bằng split() nên clean_text bị phồng vì dấu câu dính vào từ, vd 'ơi,' khác 'ơi'; sau tokenize/isalnum ở Step 3 sẽ gần cột no_punct hơn)")

clean_text            : sơn tùng m-tp: don't say 2019, i'm lạc trôi! tết x2 (oh-oh-oh)
regex của lab BBC     : s n t ng m tp don t say 2019 i m l c tr i t t x2 oh oh oh    <-- mất sạch chữ có dấu!
remove_punctuation(\w): sơn tùng m tp don t say 2019 i m lạc trôi tết x2 oh oh oh
fold_accents          : son tung m-tp: don't say 2019, i'm lac troi! tet x2 (oh-oh-oh)

Regex [^a-zA-Z0-9\s] làm mất 26.4% ký tự của corpus và băm vỡ từ tiếng Việt (vd 'lạc trôi' -> 'l c tr i').

vocab (tách theo khoảng trắng): clean_text = 16882 | no_punct = 9265 | no_accent = 12514
(vocab tính bằng split() nên clean_text bị phồng vì dấu câu dính vào từ, vd 'ơi,' khác 'ơi'; sau tokenize/isalnum ở Step 3 sẽ gần cột no_punct hơn)


**Quyết định cho retrieval lời bài hát tiếng Việt**

| Thao tác | Quyết định | Lý do |
|---|---|---|
| NFC, `html.unescape`, bỏ tag/URL, gộp khoảng trắng, chữ thường | **Làm** | Bắt buộc để cùng một chữ có cùng một token (cùng chữ "ạ" có 2 cách mã hoá Unicode; `&#039;` thành `'`). |
| Bỏ rác từng web, nhãn `[Verse]/ĐK:`, đánh số đoạn | **Làm** | Không phải lời hát; nếu giữ thì "xem", "thêm", "theo", "dõi", "tone", "capo" bị tính như từ khoá, làm lệch IDF và cho khớp sai với các truy vấn. |
| Giữ **dấu tiếng Việt** | **Giữ** trong `clean_text` | Dấu phân biệt nghĩa (*ma/má/mà/mả/mã/mạ*). Bỏ dấu là thí nghiệm riêng (`clean_no_accent`) hoặc làm ở lớp truy vấn (index cả 2 dạng) để phục vụ truy vấn gõ không dấu. |
| **Số** | Giữ | Số thường là lời thật ("1 giờ sáng", "3d") và có trong tên bài ("Intro 2022"); bỏ số làm mất truy vấn như "2019". Đánh đổi: ký hiệu lặp kiểu "x2" là nhiễu nhỏ (chỉ bị bỏ khi nằm trong dòng nhãn/header). |
| **Gạch nối** | Giữ ở `clean_text` | `m-tp` là một thực thể; tách thành `m` + `tp` khiến "tp" khớp lung tung, còn bỏ hẳn thì mất khả năng khớp `mtp`. Tách ở bước tokenize nếu cần (`oh-oh-oh` có thể coi là 1 hoặc 3 token). |
| **Dấu nháy** | Giữ | `don't` ≠ `don` + `t`; `i'm`/`i’m` (có 2 loại nháy – nên chuẩn hoá `’`→`'` ở Step 3). |
| Dấu câu khác | Giữ ở `clean_text`, bỏ ở bước tokenize (`isalnum`) | Giống lab; tránh "ơi," ≠ "ơi". Dùng `\w` Unicode, **không** dùng `[^a-zA-Z0-9\s]`. |

In [14]:
# Trùng lời SAU khi làm sạch (bắt được cả bản chỉ khác rác / dấu cách / nhãn đoạn)
d2 = corpus.copy()
d2["dup_group"] = d2.groupby("clean_text").ngroup()
sz = d2.groupby("dup_group").id.transform("size")
print("Bài thuộc nhóm trùng sau clean:", int((sz > 1).sum()), "/", len(d2),
      "| số nhóm:", d2[sz > 1].dup_group.nunique(),
      "| (trước clean – nguyên văn:", int(songs.duplicated('lyrics', keep=False).sum()), "dòng)")
corpus["dup_group"] = d2["dup_group"].values        # chỉ đánh dấu, KHÔNG xoá
corpus[["id", "site", "title", "dup_group"]].head()

Bài thuộc nhóm trùng sau clean: 238 / 2130 | số nhóm: 107 | (trước clean – nguyên văn: 234 dòng)


,id,site,title,dup_group
0,1,nhaccuatui.com,SỚM MUỘN THÌ,212
1,2,nhaccuatui.com,Pin Dự Phòng,477
2,3,nhaccuatui.com,E Là Không Thể,94
3,4,nhaccuatui.com,Em Thua Cô Ta (#1),613
4,5,nhaccuatui.com,Kho Báu,169


## Bộ truy vấn chấm điểm (10 truy vấn)

**Quy ước nhãn.** Một tài liệu là *liên quan* với truy vấn `q` nếu nó là **một phiên bản của đúng bài hát mà `q` nhắm tới** (cùng bài, khác website / khác nghệ sĩ cover / remix / lo-fi / live "Sky Tour" / "softer version"). Cách gán:

1. Chọn bài, lấy tất cả dòng có cùng *tên gốc* (bỏ phần `( … )`, `[ … ]`, `- Remix …`);
2. **Kiểm tay** từng dòng bằng độ chồng lấn lời (5-gram từ) với bản gốc và nghệ sĩ → loại các bài trùng tên nhưng khác bài;
3. **Không** coi mashup/medley là liên quan (chỉ chứa một đoạn của bài) – các dòng này được liệt kê ở cell kiểm tra bên dưới như *vùng xám*.

Loại truy vấn: `title` (tên bài), `lyric` (một câu hát), `no_accent` (gõ không dấu).

In [15]:
QUERIES = [
    # qid, type, query, relevant ids (cột `id` trong bảng songs), ghi chú
    {"qid": "q01", "type": "title",     "query": "Lạc Trôi",
     "relevant_ids": [534, 536, 1128, 1145, 1278, 1608, 2708, 2729],
     "note": "Sơn Tùng M-TP; gồm Triple D Remix & Sky Tour 2019; 4 web"},
    {"qid": "q02", "type": "title",     "query": "Chúng Ta Không Thuộc Về Nhau",
     "relevant_ids": [159, 523, 1144, 1277, 1607, 2748],
     "note": "gồm Sky Tour 2019; loại 3 mashup (2750, 2757, 2763)"},
    {"qid": "q03", "type": "title",     "query": "Thiệp Hồng Sai Tên",
     "relevant_ids": [11, 75, 117, 118, 120, 122, 123, 124],
     "note": "8 bản remix/lo-fi trên NhacCuaTui, lời gần như giống nhau"},
    {"qid": "q04", "type": "title",     "query": "Có Chàng Trai Viết Lên Cây",
     "relevant_ids": [310, 645, 774, 1030, 2611, 2612, 2672],
     "note": "Phan Mạnh Quỳnh + bản Hà Anh Tuấn (774) – cùng bài, nghệ sĩ khác"},
    {"qid": "q05", "type": "title",     "query": "Nắng Dưới Chân Mây",
     "relevant_ids": [28, 194, 195, 196, 197, 198, 199, 200, 201],
     "note": "9 phiên bản (remix/cover), chỉ NhacCuaTui; đo khả năng gom nhiều bản"},
    {"qid": "q06", "type": "lyric",     "query": "nếu ngày mai em rời xa anh",
     "relevant_ids": [537, 698, 1130, 2739, 2744],
     "note": "câu hát của 'Anh Sai Rồi'; bản hopamchuan (2744) lời ngắn/khác nên không chứa nguyên câu"},
    {"qid": "q07", "type": "lyric",     "query": "họ gọi thành phố này là nơi khó tồn tại",
     "relevant_ids": [175, 620],
     "note": "câu mở đầu 'Hà Nội' (Obito, Shiki, VSTRA) – chỉ 2 bản (ít tài liệu liên quan)"},
    {"qid": "q08", "type": "lyric",     "query": "vầng trăng treo trên núi nghiêng nghiêng",
     "relevant_ids": [204, 739, 1759],
     "note": "'Bắc Thang Lên Hỏi Ông Trời' (+ HaNoi Concert Version); loại mashup 149"},
    {"qid": "q09", "type": "no_accent", "query": "lac troi",
     "relevant_ids": [534, 536, 1128, 1145, 1278, 1608, 2708, 2729],
     "note": "cặp với q01 – đo tác động của việc gõ không dấu"},
    {"qid": "q10", "type": "no_accent", "query": "em la ai tu dau buoc den noi day",
     "relevant_ids": [42, 683, 1129, 1146, 2741],
     "note": "câu hát 'Nơi Này Có Anh' gõ không dấu; loại mashup 2757"},
]
print(len(QUERIES), "truy vấn;", Counter(q["type"] for q in QUERIES))

10 truy vấn; Counter({'title': 5, 'lyric': 3, 'no_accent': 2})


In [16]:
# ---- Kiểm tra bộ nhãn ----
cid = set(corpus.id)
by_id = corpus.set_index("id")
folded = corpus.set_index("id").clean_text.map(fold_accents)

problems = []
for q in QUERIES:
    ids = q["relevant_ids"]
    if len(set(ids)) != len(ids): problems.append((q["qid"], "id trùng trong nhãn"))
    miss = [i for i in ids if i not in cid]
    if miss: problems.append((q["qid"], f"id không có trong corpus: {miss}"))
    if len(ids) < 2: problems.append((q["qid"], "<2 tài liệu liên quan -> Recall@K không ổn định"))
print("Vấn đề:", problems if problems else "không có")

# Bảng kiểm tay: mỗi truy vấn -> các bài được gán nhãn
for q in QUERIES:
    rows = by_id.loc[q["relevant_ids"], ["site", "title", "artist"]].copy()
    rows["title"] = rows.title.map(html.unescape).str.slice(0, 55)
    rows["artist"] = rows.artist.fillna("").str.slice(0, 28)
    print(f"\n[{q['qid']}] ({q['type']}) “{q['query']}”  -> {len(rows)} bài liên quan")
    print(rows.to_string())

Vấn đề: không có

[q01] (title) “Lạc Trôi”  -> 8 bài liên quan
                  site                      title                   artist
id                                                                        
534   open.spotify.com  Lạc Trôi - Triple D Remix  Sơn Tùng M-TP, Triple D
536   open.spotify.com                   Lạc Trôi            Sơn Tùng M-TP
1128  open.spotify.com                   Lạc Trôi            Sơn Tùng M-TP
1145  open.spotify.com                   Lạc Trôi            Sơn Tùng M-TP
1278  open.spotify.com                   Lạc Trôi            Sơn Tùng M-TP
1608  open.spotify.com   Lạc Trôi - Sky Tour 2019            Sơn Tùng M-TP
2708    hopamchuan.com                   Lạc Trôi            Sơn Tùng M-TP
2729    nhaccuatui.com                   Lạc Trôi            Sơn Tùng M-TP

[q02] (title) “Chúng Ta Không Thuộc Về Nhau”  -> 6 bài liên quan
                  site                                         title         artist
id                                   

In [17]:
# ---- Vùng xám: tài liệu KHÔNG gán nhãn nhưng "trông giống" liên quan (để xác nhận quyết định loại) ----
titles_fold = corpus.set_index("id").title.map(lambda t: fold_accents(html.unescape(t).lower()))
for q in QUERIES:
    key = fold_accents(q["query"].lower())
    if q["type"] in ("title",) or q["qid"] == "q09":
        pool, where = titles_fold, "tên bài chứa cụm"          # truy vấn tên bài -> xét tiêu đề
    else:
        pool, where = folded, "lời chứa nguyên câu"            # truy vấn câu hát -> xét lời
    hit = [i for i, t in pool.items() if key in t and i not in q["relevant_ids"]]
    print(f"[{q['qid']}] “{q['query']}” – {where} nhưng không gán nhãn: {len(hit)}")
    if hit:
        print(by_id.loc[hit, ["site", "title", "artist"]].assign(title=lambda d: d.title.map(html.unescape).str.slice(0, 70),
                                                                artist=lambda d: d.artist.fillna("").str.slice(0, 30)).to_string())

# Với truy vấn lyric/no_accent: bao nhiêu tài liệu liên quan chứa NGUYÊN câu? (còn lại liên quan vì cùng bài)
print()
for q in QUERIES:
    if q["type"] in ("lyric", "no_accent") and q["qid"] != "q09":
        key = fold_accents(q["query"].lower())
        n = sum(key in folded[i] for i in q["relevant_ids"])
        print(f"[{q['qid']}] {n}/{len(q['relevant_ids'])} tài liệu liên quan chứa nguyên câu truy vấn")

[q01] “Lạc Trôi” – tên bài chứa cụm nhưng không gán nhãn: 0
[q02] “Chúng Ta Không Thuộc Về Nhau” – tên bài chứa cụm nhưng không gán nhãn: 2
                site                                                                   title          artist
id                                                                                                          
2750  hopamchuan.com             Mashup Chúng Ta Không Thuộc Về Nhau + We Don't Talk Anymore  Sơn Tùng - MTP
2763  hopamchuan.com  Mashup Chúng Ta Không Thuộc Về Nhau - Em Đã Biết - We Don't Talk Anymo        DauSmile
[q03] “Thiệp Hồng Sai Tên” – tên bài chứa cụm nhưng không gán nhãn: 0
[q04] “Có Chàng Trai Viết Lên Cây” – tên bài chứa cụm nhưng không gán nhãn: 0
[q05] “Nắng Dưới Chân Mây” – tên bài chứa cụm nhưng không gán nhãn: 0
[q06] “nếu ngày mai em rời xa anh” – lời chứa nguyên câu nhưng không gán nhãn: 0
[q07] “họ gọi thành phố này là nơi khó tồn tại” – lời chứa nguyên câu nhưng không gán nhãn: 0
[q08] “vầng trăng treo trên núi

In [18]:
with open("queries_step1-2.json", "w", encoding="utf-8") as f:
    json.dump({"db": DB_NAME, "relevance_rule": "relevant = every version of the song the query targets (same song, any site/artist/remix); mashups excluded",
               "queries": QUERIES}, f, ensure_ascii=False, indent=2)
print("Đã ghi queries_step1-2.json")

Đã ghi queries_step1-2.json


## Thảo luận
### 1. Truy vấn có từ không có trong vocabulary thì sao?
Xem trực tiếp: vocabulary xây từ `clean_text` (có dấu) và từ `clean_no_accent`.

In [19]:
vocab_acc = {w for t in corpus.clean_text for w in re.findall(r"\w+", t)}
vocab_fold = {w for t in corpus.clean_no_accent for w in re.findall(r"\w+", t)}

rows = []
for q in QUERIES:
    toks = re.findall(r"\w+", q["query"].lower())
    oov_acc = [w for w in toks if w not in vocab_acc]
    oov_fold = [w for w in re.findall(r"\w+", fold_accents(q["query"].lower())) if w not in vocab_fold]
    rows.append({"qid": q["qid"], "query": q["query"], "n_tok": len(toks),
                 "OOV (index có dấu)": oov_acc, "OOV (index bỏ dấu)": oov_fold})
pd.DataFrame(rows)

,qid,query,n_tok,OOV (index có dấu),OOV (index bỏ dấu)
0,q01,Lạc Trôi,2,[],[]
1,q02,Chúng Ta Không Thuộc Về Nhau,6,[],[]
2,q03,Thiệp Hồng Sai Tên,4,[],[]
3,q04,Có Chàng Trai Viết Lên Cây,6,[],[]
4,q05,Nắng Dưới Chân Mây,4,[],[]
5,q06,nếu ngày mai em rời xa anh,7,[],[]
6,q07,họ gọi thành phố này là nơi khó tồn tại,10,[],[]
7,q08,vầng trăng treo trên núi nghiêng nghiêng,7,[],[]
8,q09,lac troi,2,"[lac, troi]",[]
9,q10,em la ai tu dau buoc den noi day,9,"[dau, buoc, den]",[]


**Điều xảy ra khi từ truy vấn ngoài vocabulary (OOV):**

- **TF-IDF + cosine** (`TfidfVectorizer.transform`): từ lạ bị *bỏ qua im lặng*. Nếu *mọi* từ đều OOV thì vector truy vấn toàn 0 → mọi cosine = 0 → `np.argsort` trả về thứ tự gần như tuỳ ý (tài liệu index nhỏ nhất) – nhìn như "kết quả" nhưng vô nghĩa. Nếu chỉ một phần OOV, điểm chỉ dựa vào các từ còn lại (vẫn bị chuẩn hoá L2 theo vector truy vấn đã rút gọn).
- **BM25** (`BM25Okapi.get_scores`): từ lạ không có trong `doc_freqs` nên đóng góp 0 điểm; tất cả từ lạ → mọi điểm = 0. (Từ lạ **không** làm lỗi nhưng cũng không có IDF để phạt/thưởng.)
- Ở tập này OOV đến từ **bất khớp dấu** (bảng trên): `q09` ("lac troi") có **100% từ OOV** → mọi điểm = 0, P@K = R@K = 0 dù có 8 bài liên quan. `q10` có 3/9 từ OOV (`dau`, `buoc`, `den`); 6 từ còn lại (`la`, `ai`, `tu`, `noi`, `day`, `em`) tình cờ có mặt dưới dạng không dấu/tiếng Anh nên *khớp lạc đề* – tệ hơn OOV hoàn toàn vì hệ thống vẫn trả kết quả trông hợp lý. Cách xử lý: chuẩn hoá cả hai phía (index thêm bản `fold_accents`, hoặc dựng index thứ hai và hợp nhất), n-gram ký tự, sửa chính tả/fuzzy, hoặc tách từ ghép tiếng Việt cho đúng. Khi đánh giá, nên **báo riêng** các truy vấn mà toàn bộ từ OOV thay vì đếm lẫn vào trung bình, và không trả "top-K ngẫu nhiên" khi điểm đều bằng 0.
- Lưu ý khác: `BM25Okapi` không dùng IDF âm cho từ xuất hiện trong >50% tài liệu mà thay bằng `epsilon × IDF trung bình`; với lời bài hát (rất nhiều "anh", "em") nên chú ý tham số `epsilon` khi so sánh với TF-IDF.

### 2. Cần thêm nhãn / phán đoán gì để đánh giá đúng?
1. **Mức độ liên quan có thứ bậc** (0 = không, 1 = cùng bài khác bản, 2 = đúng bản gốc/nghệ sĩ chính) thay vì nhị phân – để dùng nDCG/MAP; hiện "Thiệp Hồng Sai Tên (Lo-Fi)" và bản gốc ngang hàng.
2. **ID "tác phẩm" (work id)** gom các phiên bản của một bài (kể cả remix/cover/khác tên hiển thị) → tính metric ở mức *bài hát*, tránh top-K bị chiếm bởi 5 bản copy.
3. **Nhãn cho vùng xám**: mashup/medley chứa một đoạn của bài (ví dụ id 149, 2750, 2757, 2763) là *partial relevant*? Cần quy ước và nhãn riêng.
4. **Nhãn âm cứng (hard negatives)**: cùng tên nhưng khác bài ("Tìm Em", "Xin Lỗi", "Cô Ta", "Em") và cùng nghệ sĩ khác bài – để phát hiện hệ thống khớp theo tên thay vì theo bài.
5. **Nhãn theo loại truy vấn** (tên bài / câu hát / không dấu / sai chính tả / nghệ sĩ…) để báo cáo metric theo nhóm; và **cặp truy vấn có dấu ↔ không dấu** (q01–q09) có cùng tập liên quan.
6. **Phán đoán bằng pooling** (gộp top-K của nhiều hệ thống rồi gán nhãn tay), vì ta mới gán nhãn theo tên/lời; tài liệu chưa chấm bị coi là không liên quan → có thể đánh giá thấp hệ thống. Nên có **2 người gán + độ đồng thuận (Cohen's κ)**.
7. **Nhãn chất lượng lời** (lời rỗng/rác/cắt cụt như id 2744 chỉ ~700 ký tự, 6 trang chỉ có khối "Nghệ sĩ") để tách lỗi *dữ liệu* khỏi lỗi *xếp hạng*.

### 3. Hạn chế đã biết của `clean_text`
- Nhãn người hát dạng tên riêng (`Quang Hùng MasterD:`) chỉ bị bỏ khi nằm trong `[...]` hoặc là dòng ≤3 từ kết thúc bằng `:`; nhãn nằm giữa dòng thì còn.
- Hợp Âm Chuẩn có vài bài chữ bị cách rời (`B ài hát ngày x ưa`) – lỗi nguồn, không sửa được bằng regex; ảnh hưởng tách từ.
- Bảng chữ giả Latin (`у`, `е`, `о`…) chỉ được đổi khi bài không có chữ Hàn/Nhật/Trung (để không phá lời tiếng Nga/…, nhưng với bài đa ngữ thì bỏ sót).
- Các nhãn gán tay dựa trên *tên bài + độ chồng lấn lời*; bạn nên rà lại bảng kiểm tay ở trên (đặc biệt id 2744, 620, 1608) trước khi dùng báo cáo.